In [1]:
import os 
print(os.getcwd())
os.chdir("../code/SPID_code")

print(os.getcwd())
from gmDAGGER import train_spid

/faststorage/project/GGSpeciale/GGSpeciale/simglucose
/faststorage/project/GGSpeciale/GGSpeciale/code/SPID_code


/home/elisalaegs/miniforge3/envs/thesis/lib/python3.10/site-packages/gym/envs/registration.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/home/elisalaegs/miniforge3/envs/thesis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-05 19:00:29.931283: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from stable_baselines3 import PPO
from PySRWrapper import PySRPolicy
from stable_baselines3.common.env_util import make_vec_env


In [5]:
print(os.getcwd())
os.chdir("../../simglucose")

print(os.getcwd())

/faststorage/project/GGSpeciale/GGSpeciale/code/SPID_code
/faststorage/project/GGSpeciale/GGSpeciale/simglucose


In [ ]:
import warnings
from stable_baselines3 import PPO

import gymnasium as gym
import numpy as np
from simglucose_env import make_simglucose_spid_env

from simglucose_env import MultiPatientSimglucoseEnv

environment = lambda: MultiPatientSimglucoseEnv(
    patient_names=[
        "adult#001",
        "adult#002",
        "adult#003",
        "adult#004",
        "adult#005",
    ],
    env_id="simglucose-distill",
    max_episode_steps=480,
    normalize=True,
    scenario_mode="fixed_hb",
    use_custom_reward=True,
)

warnings.filterwarnings(
    "ignore",
    message="You are trying to run PPO on the GPU"
)

warnings.filterwarnings(
    "ignore",
    message="Note: it looks like you are running in Jupyter"
)

# example

# rewards, best_policy, wrapper, run_dir = train_spid(r"/home/ashc/GGSpeciale/ashc_repo/GGSpeciale/code/baseline_code/baseline_models/cartpole/PPO_cartpole.zip", PPO, "", "CartPole-v1", n_iter=1, total_timesteps=3, verbose=2)

teacher_model_path = "output/models/best/best_model.zip"
teacher_model = PPO

rewards, best_policy, wrapper, run_dir = train_spid(teacher_path = teacher_model_path, 
                                                    teacher_model = teacher_model, 
                                                    save_folder_path="distil_results", 
                                                    save_results=True, 
                                                    environment=environment, 
                                                    n_iter=12, 
                                                    total_timesteps=12*1000, 
                                                    n_eval_episodes=10,
                                                    verbose=2)

Loaded local teacher from: output/models/best/best_model.zip
finished collecting trajectories
computing advantages
training
Evaluating trained model
Iteration 0: student reward = -1354.7163 +/- 545.2932
Loaded local teacher from: output/models/best/best_model.zip
finished collecting trajectories
computing advantages
training
Evaluating trained model


In [3]:
student = PySRPolicy.load(r"/home/elisalaegs/GGSpeciale/GGSpeciale/simglucose/distil_results/best_student_policy.joblib")
best_policy = student
for i, policy in enumerate(best_policy.policy_list):
    print(f"\n=== Action dimension {i} ===")

    sr = policy.sr
    
    best_eq = sr.get_best()["equation"]
    print("\nBest equation:")
    print(best_eq)

Policy loaded

=== Action dimension 0 ===

Best equation:
(-0.960025 / (x0 + -0.671842)) * 0.009929599


In [4]:
student = PySRPolicy.load(r"/home/elisalaegs/GGSpeciale/GGSpeciale/simglucose/distil_results/best_student_policy.joblib")
best_policy = student
for i, policy in enumerate(best_policy.policy_list):
    print(f"\n=== Action dimension {i} ===")

    sr = policy.sr
    
    best_eq = sr.get_best()["equation"]
    print("\nBest equation:")
    print(best_eq)

Policy loaded

=== Action dimension 0 ===

Best equation:
square(square(square(square(exp(((x0 - (x2 * 2.2867436)) - 0.3107938) - x1)))))


In [ ]:
def evaluate_gm_dagger(
    student_policy,
    teacher_policy,
    env,
    seed=0,
    max_steps=2000,
    gamma=0.99,
    alpha=1e-8,
    eps=1e-8,
    deterministic=True,
):
    """
    Roll out the student policy and compare it to the teacher policy.

    Returns
    -------
    results : dict
        {
            "total_reward": float,
            "mean_performance_loss": float,
            "mean_fidelity_gap": float,
            "mean_gm_dagger_loss": float,
            "rewards": np.ndarray,
            "student_actions": np.ndarray,
            "teacher_actions": np.ndarray,
            "observations": np.ndarray,
            "performance_losses": np.ndarray,
            "fidelity_gaps": np.ndarray,
            "gm_dagger_losses": np.ndarray,
        }
    """

    obs, info = env.reset(seed=seed)

    rewards = []
    observations = []
    student_actions = []
    teacher_actions = []
    next_observations = []

    terminated = False
    truncated = False

    for _ in range(max_steps):
        # Student action: action used in the environment
        student_action, _ = student_policy.predict(obs, deterministic=deterministic)

        # Teacher action: oracle action at the same state
        teacher_action, _ = teacher_policy.predict(obs, deterministic=deterministic)

        student_action = np.asarray(student_action, dtype=np.float32)
        teacher_action = np.asarray(teacher_action, dtype=np.float32)

        # Fix common VecEnv / PySR shapes
        if student_action.ndim == 2 and student_action.shape[0] == 1:
            student_action = student_action[0]
        if teacher_action.ndim == 2 and teacher_action.shape[0] == 1:
            teacher_action = teacher_action[0]

        next_obs, reward, terminated, truncated, info = env.step(student_action)

        observations.append(np.asarray(obs).copy())
        next_observations.append(np.asarray(next_obs).copy())
        student_actions.append(student_action.copy())
        teacher_actions.append(teacher_action.copy())
        rewards.append(float(reward))

        obs = next_obs

        if terminated or truncated:
            break

    observations = np.asarray(observations)
    next_observations = np.asarray(next_observations)
    student_actions = np.asarray(student_actions)
    teacher_actions = np.asarray(teacher_actions)
    rewards = np.asarray(rewards, dtype=np.float32)

    # Fidelity gap: || pi_student(s) - pi_teacher(s) ||^2
    fidelity_gaps = np.sum((student_actions - teacher_actions) ** 2, axis=-1)

    # Performance loss approximation.
    # If you have access to teacher value function, use TD-style advantage.
    performance_losses = compute_teacher_advantage_loss(
        teacher_policy,
        observations,
        student_actions,
        rewards,
        next_observations,
        gamma=gamma,
    )

    # Ensure positive values for geometric mean
    gp = np.abs(performance_losses)# + alpha
    gf = fidelity_gaps# + eps

    gm_dagger_losses = gp * gf

    return {
        "total_reward": float(np.sum(rewards)),
        "mean_performance_loss": float(np.mean(gp)),
        "mean_fidelity_gap": float(np.mean(gf)),
        "mean_gm_dagger_loss": float(np.mean(gm_dagger_losses)),
        "rewards": rewards,
        "student_actions": student_actions,
        "teacher_actions": teacher_actions,
        "observations": observations,
        "performance_losses": gp,
        "fidelity_gaps": gf,
        "gm_dagger_losses": gm_dagger_losses,
    }

import torch


def compute_teacher_advantage_loss(
    teacher,
    states,
    actions,
    rewards,
    next_states,
    gamma=0.99,
):
    """
    Approximate performance loss using the teacher's critic/value function.

    For PPO/TRPO/A2C:
        loss ≈ |r + gamma V(s') - V(s)|

    For SAC/TD3/DDPG:
        loss ≈ |Q(s,a_student) - [r + gamma Q(s', pi_teacher(s'))]|
    """

    device = torch.device("cpu")
    teacher.policy = teacher.policy.to(device)

    states_t = torch.as_tensor(states, dtype=torch.float32, device=device)
    actions_t = torch.as_tensor(actions, dtype=torch.float32, device=device)
    rewards_t = torch.as_tensor(rewards, dtype=torch.float32, device=device).view(-1)
    next_states_t = torch.as_tensor(next_states, dtype=torch.float32, device=device)

    if actions_t.ndim == 1:
        actions_t = actions_t.unsqueeze(-1)

    with torch.no_grad():
        # PPO / TRPO / A2C
        if hasattr(teacher.policy, "predict_values"):
            v_s = teacher.policy.predict_values(states_t).squeeze(-1)
            v_sp = teacher.policy.predict_values(next_states_t).squeeze(-1)

            td_error = rewards_t + gamma * v_sp - v_s
            return np.abs(td_error.cpu().numpy())

        # SAC / TD3 / DDPG
        elif hasattr(teacher.policy, "critic"):
            q_s = teacher.policy.critic.q1_forward(states_t, actions_t).squeeze(-1)

            next_actions_np, _ = teacher.predict(
                next_states,
                deterministic=True,
            )
            next_actions_t = torch.as_tensor(
                next_actions_np,
                dtype=torch.float32,
                device=device,
            )

            if next_actions_t.ndim == 1:
                next_actions_t = next_actions_t.unsqueeze(-1)

            q_sp = teacher.policy.critic.q1_forward(
                next_states_t,
                next_actions_t,
            ).squeeze(-1)

            td_error = q_s - (rewards_t + gamma * q_sp)
            return np.abs(td_error.cpu().numpy())

        else:
            # Fallback: no value/critic available
            return np.abs(rewards - np.mean(rewards))

env = HoverActionShapeWrapper(
    HoverAviary(
        obs=DEFAULT_OBS,
        act=DEFAULT_ACT,
        gui=False,
        record=False,
    )
)

#teacher = PPO.load(r"C:\GitHub\GGSpeciale\quadcopter-suite\results\save-04.10.2026_15.39.13\best_model.zip")
#student = PySRPolicy.load(r"C:\GitHub\GGSpeciale\quadcopter-suite\distil-results_1000x12\best_student_policy.joblib")
#student_stupid = PySRPolicy.load(r"C:\GitHub\GGSpeciale\quadcopter-suite\distil-results_1000x12_no_Q\best_student_policy.joblib")

results = evaluate_gm_dagger(
    student_policy=student_stupid,
    teacher_policy=teacher,
    env=env,
    seed=0,
    max_steps=20000,
)

env.close()

print("Total reward:", results["total_reward"])
print("Performance loss:", results["mean_performance_loss"])
print("Fidelity gap:", results["mean_fidelity_gap"])
print("GM-DAGGER loss:", results["mean_gm_dagger_loss"])